#TRABAJO PRACTICO: búsqueda voraz y A* sobre un grafo dirigido


**Estudiante:** JUSTIN GARCIA

**Propósito:** Implementar tres estrategias de búsqueda sobre el grafo dirigido de la clase —costo uniforme (UCS), voraz por el mejor primero y A— y
comparar qué camino devuelven

# El grafo dirigido, Estructura de datos y esquema común

In [ ]:
import heapq, itertools


# generamos un diccionario de las adyacencias de cada nodo
GRAFO = {
    "S": [("A", 2), ("B", 2)],
    "A": [("C", 2), ("D", 5)],
    "B": [("D", 2)],
    "C": [("G", 3)],
    "D": [("G", 6)],
    "G": [],
}

H = {
    "S": 7,
    "A": 5,
    "B": 7,
    "C": 3,
    "D": 6,
    "G": 0,
}

OBJETIVO = "G"
INICIO = "S"

#funcion general
def crear_nodo(estado, padre=None, accion=None, g=0, h=0):
    return {
        "estado": estado,
        "padre": padre,
        "accion": accion,
        "g": g,
        "h": h,
        "f": g + h,
    }

def reconstruir_camino(nodo):
  camino = []
  actual = nodo
  while actual is not None:
    camino.append(actual["estado"])
    actual = actual["padre"]
  camino.reverse()
  return camino


# Requerimientos de implementación y algoritmos implementados


In [65]:
# Esqueleto comun de busqueda

def buscar(grafo, h, inicio, objetivo, clave_prioridad, nombre_algoritmo=""):
  contador = itertools.count()
  frontera = []
  raiz = crear_nodo(inicio, g=0, h=h[inicio])
  heapq.heappush(frontera, (clave_prioridad(raiz), next(contador), raiz))

# registros que se iran actualizando (dados en la consigna)
  mejor_g = {inicio: 0}
  generados = 1
  expandidos = 0
  frontera_maxima = 1
  reaperturas = 0
  traza = []

  while frontera:
      prioridad, orden, nodo = heapq.heappop(frontera)

      # descartamos si es una entrada obsoleta
      if nodo["g"] > mejor_g[nodo["estado"]]:
        continue


    # guardar la traza para saber que se extrajo y como quedaria la frontera
      frontera_legible = [(n["estado"], p) for p, o, n in frontera]
      traza.append({
            "extraido": nodo["estado"],
            "g": nodo["g"],
            "h": nodo["h"],
            "f": nodo["f"],
            "prioridad": prioridad,
            "frontera": frontera_legible,
        })

      if nodo["estado"] == objetivo:
            return {
                "algoritmo": nombre_algoritmo,
                "camino": reconstruir_camino(nodo),
                "costo": nodo["g"],
                "generados": generados,
                "expandidos": expandidos,
                "frontera_maxima": frontera_maxima,
                "reaperturas": reaperturas,
                "traza": traza,
            }

       # a partir de acá el nodo se considera "expandido": se le generan sucesores
      expandidos += 1

    # relajar sucesores
      for vecino, costo in grafo[nodo["estado"]]:
            nuevo_g = nodo["g"] + costo
            es_nuevo = vecino not in mejor_g
            es_mejora = (not es_nuevo) and nuevo_g < mejor_g[vecino]

            if es_nuevo or es_mejora:
                if es_mejora:
                    reaperturas += 1
                mejor_g[vecino] = nuevo_g
                nuevo_nodo = crear_nodo(
                    vecino, padre=nodo,
                    accion=f"{nodo['estado']} -> {vecino}",
                    g=nuevo_g, h=h[vecino],
                )
                heapq.heappush(frontera, (clave_prioridad(nuevo_nodo), next(contador), nuevo_nodo))
                generados += 1

      frontera_maxima = max(frontera_maxima, len(frontera))

  return None  # fracaso

# implementacion de los tres algoritmos (solo cambia la clave de prioridad)

def ucs(grafo, h, inicio, objetivo):
    return buscar(grafo, h, inicio, objetivo, lambda n: n["g"], "UCS")


def voraz(grafo, h, inicio, objetivo):
    return buscar(grafo, h, inicio, objetivo, lambda n: n["h"], "Voraz")


def a_estrella(grafo, h, inicio, objetivo):
    return buscar(grafo, h, inicio, objetivo, lambda n: n["f"], "A*")


# Impresión de trazas

def imprimir_traza(resultado):
    print(f"\n--- Traza de {resultado['algoritmo']} ---")
    for i, paso in enumerate(resultado["traza"], start=1):
        print(f"  Paso {i}: se extrae '{paso['extraido']}' "
              f"(g={paso['g']}, h={paso['h']}, f={paso['f']}, prioridad={paso['prioridad']}) "
              f"| frontera restante: {paso['frontera']}")
    print(f"  Camino: {' -> '.join(resultado['camino'])}")
    print(f"  Costo total: {resultado['costo']}")
    print(f"  Generados: {resultado['generados']} | "
          f"Expandidos: {resultado['expandidos']} | "
          f"Frontera máxima: {resultado['frontera_maxima']} | "
          f"Reaperturas: {resultado['reaperturas']}")


resultado_ucs = ucs(GRAFO, H, INICIO, OBJETIVO)
resultado_voraz = voraz(GRAFO, H, INICIO, OBJETIVO)
resultado_astar = a_estrella(GRAFO, H, INICIO, OBJETIVO)

for resultado in (resultado_ucs, resultado_voraz, resultado_astar):
    imprimir_traza(resultado)



--- Traza de UCS ---
  Paso 1: se extrae 'S' (g=0, h=7, f=7, prioridad=0) | frontera restante: []
  Paso 2: se extrae 'A' (g=2, h=5, f=7, prioridad=2) | frontera restante: [('B', 2)]
  Paso 3: se extrae 'B' (g=2, h=7, f=9, prioridad=2) | frontera restante: [('C', 4), ('D', 7)]
  Paso 4: se extrae 'C' (g=4, h=3, f=7, prioridad=4) | frontera restante: [('D', 4), ('D', 7)]
  Paso 5: se extrae 'D' (g=4, h=6, f=10, prioridad=4) | frontera restante: [('D', 7), ('G', 7)]
  Paso 6: se extrae 'G' (g=7, h=0, f=7, prioridad=7) | frontera restante: []
  Camino: S -> A -> C -> G
  Costo total: 7
  Generados: 7 | Expandidos: 5 | Frontera máxima: 3 | Reaperturas: 1

--- Traza de Voraz ---
  Paso 1: se extrae 'S' (g=0, h=7, f=7, prioridad=7) | frontera restante: []
  Paso 2: se extrae 'A' (g=2, h=5, f=7, prioridad=5) | frontera restante: [('B', 7)]
  Paso 3: se extrae 'C' (g=4, h=3, f=7, prioridad=3) | frontera restante: [('D', 6), ('B', 7)]
  Paso 4: se extrae 'G' (g=7, h=0, f=7, prioridad=0) | fron

# Verificación y comparación

In [ ]:
def imprimir_tabla(resultados):
    print("\n--- Tabla comparativa ---")
    claves_prioridad = {"UCS": "g", "Voraz": "h", "A*": "g+h"}
    encabezado = f"{'Resultado':<28}" + "".join(f"{r['algoritmo']:<12}" for r in resultados)
    print(encabezado)
    print(f"{'Camino':<28}" + "".join(f"{'-'.join(r['camino']):<12}" for r in resultados))
    print(f"{'Costo':<28}" + "".join(f"{r['costo']:<12}" for r in resultados))
    print(f"{'Prioridad':<28}" + "".join(f"{claves_prioridad[r['algoritmo']]:<12}" for r in resultados))
    print(f"{'Expandidos antes de G':<28}" + "".join(f"{r['expandidos']:<12}" for r in resultados))

imprimir_tabla([resultado_ucs, resultado_voraz, resultado_astar])




--- Tabla comparativa ---
Resultado                   UCS         Voraz       A*          
Camino                      S-A-C-G     S-A-C-G     S-A-C-G     
Costo                       7           7           7           
Prioridad                   g           h           g+h         
Expandidos antes de G       5           3           3           


| Resultado                  | UCS      | Voraz    | A*       |
|----------------------------|----------|----------|----------|
| Camino                     | S-A-C-G  | S-A-C-G  | S-A-C-G  |
| Costo                      | 7        | 7        | 7        |
| Prioridad                  | g        | h        | g+h      |
| Expandidos antes de extraer G | 5     | 3        | 3        |
| Estados Generados          | 7        | 6        | 6        |
| Frontera Máxima            | 3        | 3        | 3        |
| Reaperturas                | 1        | 0        | 0        |

# Preguntas de análisis


### 1. ¿Por qué voraz y A* coinciden en este grafo?

La heurística dada es **consistente** y, además, coincide exactamente con
el costo real restante a lo largo del camino óptimo: `h(S)=7` y el costo
real S→G es 7; `h(A)=5` y el costo real A→G es 2+3=5; `h(C)=3` y el costo
real C→G es 3. Al ser una estimación perfecta sobre esa rama, la prioridad
`f=g+h` de A* se mantiene **constante (=7)** en cada paso del camino
óptimo. La rama alternativa (B, D) tiene valores de `h` más altos que
reflejan correctamente que es más cara. Como ninguna rama "engaña" a la
heurística, voraz (que solo mira `h`) termina tomando la misma decisión
que A* (que pondera `g+h`) en cada bifurcación.

### 2. ¿Garantiza voraz devolver el camino de menor costo en general?

**No.** Su prioridad es únicamente `h(n)`: ignora por completo el costo ya
recorrido (`g`). Esto significa que puede preferir un estado que "parece"
cercano al objetivo aunque llegar hasta ahí haya sido muy caro, y descartar
una alternativa con mayor `h` pero con un costo real total menor. Al no
ponderar el gasto ya hecho, la prioridad de voraz no es una cota del costo
total del camino, por eso no puede garantizar optimalidad como sí lo hace
A* con una heurística admisible.

### 3. ¿Qué ocurre si se usa h = 0 en A*?

La prioridad queda `f = g + 0 = g`, es decir, **A* pasa a ordenar la
frontera exactamente igual que UCS**. Con `h=0`, A* y costo uniforme son
el mismo algoritmo.

### 4. ¿Hubo reaperturas en UCS? ¿Y en A* y voraz? ¿Por qué?

En esta ejecución, **UCS tuvo 1 reapertura** (mejoró el `g` de D al
encontrar el camino S→B→D, más barato que el que ya tenía por S→A→D).
Voraz y A* tuvieron 0. La razón es que UCS no tiene ninguna guía sobre
qué rama conviene explorar primero, así que puede alcanzar un mismo estado
por dos caminos distintos antes de decidirse por el objetivo. Voraz y A*,
guiados por `h`, nunca llegaron a expandir la rama de B lo suficiente como
para generar una segunda vía hacia D, porque encontraron el objetivo antes
de necesitarlo.

### 5. ¿"Expandir menos estados" significa "camino más barato"?

**No necesariamente**; son dos cosas distintas. "Expandir menos" habla de
la eficiencia de la búsqueda (cuánto trabajo hizo el algoritmo), mientras
que el costo del camino depende de qué garantía ofrece su prioridad. Acá
UCS expandió más estados (5) que voraz y A* (3) y aun así los tres
llegaron al mismo costo óptimo (7): UCS lo garantiza **por construcción**
(siempre expande en orden de `g` creciente), mientras que voraz llegó al
mismo resultado por cómo está construido este grafo en particular, no por
una garantía general. En otro grafo, voraz podría expandir menos estados y
aun así devolver un camino más caro que el óptimo.